# einops-einsum — worked example 1: Trace of a square matrix via repeated index

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-einsum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Concept

An einsum index that appears **twice in the same operand** selects the diagonal of those two axes. Reusing one letter for both axes of a square matrix, then omitting it from the output, sums the diagonal entries — that is the matrix trace. No second tensor and no `.diagonal()` call are needed.

## Worked solution

We want `trace(A) = sum_i A[i, i]` for a square `(n, n)` matrix.

1. **Name both axes the same letter.** Writing the input pattern as `'i i'` tells einsum that the row index and the column index are the *same* index `i`. This is the diagonal-selection rule: `A[i, i]` for each `i`.
2. **Omit the index from the output.** The output side `-> ` is empty (a scalar). Because `i` appears on the input but not on the output, einsum sum-contracts over it — exactly `sum_i A[i, i]`.
3. **Result is a 0-D tensor.** The whole pattern `'i i ->'` collapses the diagonal to a single scalar. This matches `torch.trace(A)` / `A.diagonal().sum()`.

The key idea: repeating a letter *within one operand* is diagonal extraction, and omitting it is reduction — combine them to get the trace in one pattern.

In [ ]:
def trace_via_einsum(A: Tensor) -> Tensor:
    return einsum(A, 'i i ->')


t.manual_seed(0)
A = t.randn(5, 5)
out = trace_via_einsum(A)
print('trace      =', out.item())
print('reference  =', t.trace(A).item())